imports:

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import os

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA, NMF
from sklearn.datasets import make_blobs
from IPython.display import Audio, display
from pathlib import Path

In [8]:
df = pd.read_csv('labels_new.csv')

display(df.head())
display(df['genre'].value_counts())
print("aantal samples:",len(df))
print("aantal genres:",df['genre'].nunique())

,filename,genre
0,m00248.wav,metal
1,m00230.wav,country
2,m00637.wav,hiphop
3,m00627.wav,metal
4,m00138.wav,reggae


genre
metal        5
country      5
hiphop       5
reggae       5
classical    5
jazz         5
rock         5
pop          5
blues        5
disco        5
Name: count, dtype: int64

aantal samples: 50
aantal genres: 10


In [9]:
labeled_audio_dir = Path('labeled')
labeled_audio_files = os.listdir(labeled_audio_dir)

In [12]:
# https://librosa.org/doc/latest/feature.html

class LibrosaFeatures:

    def __init__(self, audio_dir):
        self.audio_dir = Path(audio_dir)
        self.features_list = []

    def features(self, audio_files):
        self.features_list = [] 
        for file in audio_files:
            file_path = self.audio_dir / file
            features = self.process_file(file_path, file)
            self.features_list.append(features)
        return pd.DataFrame(self.features_list)

    def process_file(self, file_path, file):
        y, sr = librosa.load(file_path)

        features = {
            'filename': file,
            'spectral_bandwidth': np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr)),
            'spectral_bandwidth_std': np.std(librosa.feature.spectral_bandwidth(y=y, sr=sr)),
            'spectral_centroid': np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)),
            'spectral_centroid_std': np.std(librosa.feature.spectral_centroid(y=y, sr=sr)),
            'zero_crossing_rate': np.mean(librosa.feature.zero_crossing_rate(y)),
            'zero_crossing_rate_std': np.std(librosa.feature.zero_crossing_rate(y)),
            'rms': np.mean(librosa.feature.rms(y=y)),
            'rms_std': np.std(librosa.feature.rms(y=y)),
            'rolloff': np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr)),
            'flatness': np.mean(librosa.feature.spectral_flatness(y=y)),
            'tempo': librosa.beat.tempo(y=y, sr=sr)[0]


        }

        # Rhythm
        rhythm = librosa.onset.onset_strength(y=y, sr=sr)
        tempo, beats = librosa.beat.beat_track(onset_envelope=rhythm, sr=sr)

        features.update({
            'onset_strength_mean': np.mean(rhythm),
            'onset_strength_std': np.std(rhythm),
            'beat_strength': np.mean(librosa.util.normalize(rhythm)[beats]) if len(beats) > 0 else 0
        })

        # Harmonic
        harmonic, percussive = librosa.effects.hpss(y)
        features.update({
            'harmonic_ratio': np.mean(np.abs(harmonic)) / (np.mean(np.abs(percussive)) + 1e-8),
            'percussive_ratio': np.mean(np.abs(percussive)) / (np.mean(np.abs(harmonic)) + 1e-8)
        })

        # MFCC
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        for i in range(13):
            features[f'mfcc_{i+1}'] = np.mean(mfcc[i])
            features[f'mfcc_{i+1}_std'] = np.std(mfcc[i])

        # Chromagram
        chromagram = librosa.feature.chroma_stft(y=y, sr=sr)
        for i in range(12):
            features[f'chroma_{i+1}'] = np.mean(chromagram[i])
            features[f'chroma_{i+1}_std'] = np.std(chromagram[i])

        # Tempogram
        tempogram = librosa.feature.tempogram(y=y, sr=sr)
        features.update({
            'tempogram_ratio': np.mean(tempogram) / (np.std(tempogram) + 1e-8),
            'tempogram_std': np.std(tempogram)
        })

        return features

    def merge(self, labels_df):
        features_df = pd.DataFrame(self.features_list)
        return pd.merge(features_df, labels_df, on='filename')

import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=FutureWarning)
    pull = LibrosaFeatures(labeled_audio_dir)
    df_features = pull.features(labeled_audio_files)
    df_audio_features = pull.merge(df)



Ik heb geprobeerd met de nieuwe code te doen maar dat lukt niet dan blijf ik een error krijgen. Er gaat waarschijnlijk iets fout in de pip install van mijn laptop maar dat is dan maar zo. Ik heb de warnings laten removen.

In [13]:
df_audio_features.columns

Index(['filename', 'spectral_bandwidth', 'spectral_bandwidth_std',
       'spectral_centroid', 'spectral_centroid_std', 'zero_crossing_rate',
       'zero_crossing_rate_std', 'rms', 'rms_std', 'rolloff', 'flatness',
       'tempo', 'onset_strength_mean', 'onset_strength_std', 'beat_strength',
       'harmonic_ratio', 'percussive_ratio', 'mfcc_1', 'mfcc_1_std', 'mfcc_2',
       'mfcc_2_std', 'mfcc_3', 'mfcc_3_std', 'mfcc_4', 'mfcc_4_std', 'mfcc_5',
       'mfcc_5_std', 'mfcc_6', 'mfcc_6_std', 'mfcc_7', 'mfcc_7_std', 'mfcc_8',
       'mfcc_8_std', 'mfcc_9', 'mfcc_9_std', 'mfcc_10', 'mfcc_10_std',
       'mfcc_11', 'mfcc_11_std', 'mfcc_12', 'mfcc_12_std', 'mfcc_13',
       'mfcc_13_std', 'chroma_1', 'chroma_1_std', 'chroma_2', 'chroma_2_std',
       'chroma_3', 'chroma_3_std', 'chroma_4', 'chroma_4_std', 'chroma_5',
       'chroma_5_std', 'chroma_6', 'chroma_6_std', 'chroma_7', 'chroma_7_std',
       'chroma_8', 'chroma_8_std', 'chroma_9', 'chroma_9_std', 'chroma_10',
       'chroma_10_st

# Opdracht 2

In [14]:
unlabeled_audio_dir = Path('unlabeled')
unlabeled_audio_files = os.listdir(unlabeled_audio_dir)

df_unlabeled = pd.DataFrame(os.listdir(unlabeled_audio_dir))